# GPU-Accelerated Coin Counting — PUSL2101 Coursework

**Task:** Object counting (coins) from images.

**Pipeline in this notebook:**
1. Preprocessing
2. Classical computer-vision detector (feature extraction + Hough Circle Transform) — a fast, training-free baseline
3. Auto-labelling: turn the classical detector's output into YOLO-format bounding-box labels (since the raw dataset has no annotations)
4. Data augmentation to grow the small (<100 image) dataset
5. GPU-accelerated deep-learning detector: fine-tune YOLOv8n (Ultralytics/PyTorch, CUDA) on the augmented, auto-labelled set
6. Evaluation: classical vs. deep-learning counts vs. your own manual ground truth

**Running locally in VS Code:** see `README.md` for environment setup (venv + `pip install -r requirements.txt`).
If you have an NVIDIA GPU with CUDA drivers installed, training will use it automatically; otherwise it falls back to CPU (slower, but the notebook still runs end to end — just cut `epochs` down in Section 7 if training on CPU).


In [1]:
import cv2
import numpy as np
import os, glob, shutil, random, json
import matplotlib.pyplot as plt
import torch

print("OpenCV:", cv2.__version__)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected - training will run on CPU (Section 7 will be slow).")

random.seed(42)
np.random.seed(42)


OpenCV: 5.0.0
Torch: 2.11.0+cu128 | CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 1. Load the dataset

Put all your raw coin images in one local folder and point `DATA_DIR` at it.


In [2]:
# EDIT THIS to the local folder containing your coin images
DATA_DIR = r"./data/coin_images"

image_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.jpg")) +
                      glob.glob(os.path.join(DATA_DIR, "*.jpeg")) +
                      glob.glob(os.path.join(DATA_DIR, "*.png")))
print(f"Found {len(image_paths)} images in {DATA_DIR}")
assert len(image_paths) > 0, "No images found - check DATA_DIR"

# quick look at a few
fig, axes = plt.subplots(1, min(4, len(image_paths)), figsize=(16, 4))
for ax, p in zip(np.atleast_1d(axes), image_paths[:4]):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(os.path.basename(p)); ax.axis('off')
plt.tight_layout(); plt.show()


NameError: name 'glob' is not defined

## 2. Preprocessing

Standard pipeline: resize to a consistent working size (keeps Hough parameters
comparable across images), convert to grayscale, and denoise with a median blur
(median blur preserves circular edges better than Gaussian blur here, and is
robust to the salt-and-pepper texture of coin engravings).


In [ ]:
def preprocess(img, target_max_dim=1000):
    """Resize + grayscale + denoise. Returns (resized_bgr, gray_blurred)."""
    h, w = img.shape[:2]
    scale = target_max_dim / max(h, w)
    img_r = cv2.resize(img, (int(w * scale), int(h * scale)))
    gray = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)
    blurred = cv2.medianBlur(gray, 7)
    return img_r, blurred

# demo
demo_img = cv2.imread(image_paths[0])
demo_resized, demo_gray = preprocess(demo_img)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(demo_resized, cv2.COLOR_BGR2RGB)); axes[0].set_title("Resized"); axes[0].axis('off')
axes[1].imshow(demo_gray, cmap='gray'); axes[1].set_title("Grayscale + median blur"); axes[1].axis('off')
plt.tight_layout(); plt.show()


## 3. Classical CV detector — feature extraction + Hough Circle Transform

**Why Hough Circles and not a simple brightness threshold:** coins in this dataset
range from near-black to silver to gold, and some are nearly the same brightness
as the background, so a single global/Otsu threshold fails to separate several
coins from the background (tested and confirmed on this dataset). Coins are
reliably circular though, so working from **edges** (Hough Circle Transform)
rather than **brightness** is far more robust here.

We add non-maximum suppression (NMS) on the returned circles as a feature-based
cleanup step: Hough sometimes fires two overlapping circle hypotheses on the
same coin (especially the engraved/textured ones), so we merge any two detected
circles whose centres are closer than 0.65× the sum of their radii.


In [ ]:
def nms_circles(circles, overlap_thresh=0.65):
    """Greedy NMS: circles are already ordered by Hough accumulator strength.
    Drop any circle whose centre is too close to an already-kept circle."""
    keep = []
    for (x, y, r) in circles:
        is_dup = False
        for (kx, ky, kr) in keep:
            d = np.hypot(x - kx, y - ky)
            if d < overlap_thresh * (r + kr):
                is_dup = True
                break
        if not is_dup:
            keep.append((x, y, r))
    return keep


def detect_coins_classical(img, min_radius=15, max_radius=60, return_vis=False):
    """Detect coins with Hough Circle Transform on a preprocessed image.
    Returns list of (x, y, r) in the coordinate system of the resized image,
    and (optionally) an annotated visualisation image.
    """
    img_r, gray = preprocess(img)

    circles = cv2.HoughCircles(
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=45,
        param1=60, param2=32, minRadius=min_radius, maxRadius=max_radius
    )

    detections = []
    if circles is not None:
        raw = np.round(circles[0]).astype(int)
        detections = nms_circles(raw)

    if not return_vis:
        return detections, img_r

    vis = img_r.copy()
    for i, (x, y, r) in enumerate(detections, start=1):
        cv2.circle(vis, (x, y), r, (0, 255, 0), 3)
        cv2.putText(vis, str(i), (x - 10, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    return detections, vis


# demo on a handful of images
sample = image_paths[:6]
fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 5))
for ax, p in zip(np.atleast_1d(axes), sample):
    img = cv2.imread(p)
    dets, vis = detect_coins_classical(img, return_vis=True)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{os.path.basename(p)}\ncount = {len(dets)}")
    ax.axis('off')
plt.tight_layout(); plt.show()


## 4. Baseline evaluation (classical CV vs. your manual count)

Fill in `ground_truth` below with the true coin count for each filename
(count them yourself once — this is your "labels" for evaluation, not for training).


In [ ]:
ground_truth = {
    # "IMG_0001.jpg": 7,
    # "IMG_0002.jpg": 17,
    # ... fill in for as many images as you're willing to hand-count
}

def evaluate_counts(pred_counts: dict, gt: dict):
    rows, errors = [], []
    for fname, gt_count in gt.items():
        pred = pred_counts.get(fname)
        if pred is None:
            continue
        err = pred - gt_count
        errors.append(abs(err))
        rows.append((fname, gt_count, pred, err))
    mae = np.mean(errors) if errors else float('nan')
    exact_acc = np.mean([e == 0 for e in errors]) if errors else float('nan')
    return rows, mae, exact_acc

classical_counts = {}
for p in image_paths:
    dets, _ = detect_coins_classical(cv2.imread(p))
    classical_counts[os.path.basename(p)] = len(dets)

rows, mae, acc = evaluate_counts(classical_counts, ground_truth)
print(f"{'file':30s} {'GT':>4s} {'pred':>5s} {'err':>5s}")
for fname, gtc, pred, err in rows:
    print(f"{fname:30s} {gtc:4d} {pred:5d} {err:5d}")
print(f"\nClassical CV baseline — MAE: {mae:.2f}, Exact-match accuracy: {acc*100:.1f}%")
